# Lab 6.4 &mdash; Document Loading and Splitting

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Load a real file off disk with <code>TextLoader</code> and see what a <code>Document</code> is
- Split it with <code>RecursiveCharacterTextSplitter</code> and choose the two numbers that matter
- Watch a rule get separated from the exception that qualifies it &mdash; and put it back
- See what overlap actually buys, at the boundary where it matters

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a
> `Document`, a Chroma collection, a retriever, a chain), so they are deterministic and do not
> depend on the chat model. Cells marked **Run it for real** put your code in front of the
> sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **Two different models are in play, and only one of them is billed.** The **chat model**
> (`qwen36-35b-a3b-lab`) answers questions and is reached over the gateway. The **embedding
> model** (`all-MiniLM-L6-v2`, 384 dimensions) turns text into vectors and runs on this pod's
> own CPU &mdash; no key, no gateway, no tokens. Keeping them straight is most of Module 6.

> **This lab makes no gateway calls.** Splitting is arithmetic on strings; the model
> never sees any of it. It also decides more about retrieval quality than the model does.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap, warnings
from typing import Any, Callable

warnings.filterwarnings("ignore")     # sentence-transformers is chatty on first import

WORK = os.path.join("/tmp", "awmas-lab-6-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the CHAT model: qwen, through the sandbox gateway -------------------
# Already configured -- nothing to install, no key to register. Read from the
# environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Chat model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- the EMBEDDING model: local, free, nothing to configure --------------
# all-MiniLM-L6-v2, 384 dimensions. It runs on this pod's CPU and has nothing to do with
# the chat model above: no gateway, no key, no tokens billed. The cache is already warm
# in your sandbox, so the first call is a second or two, not a download.
#
# It is reached through onnxruntime rather than torch, and that is a measured choice
# rather than a taste: same model, same vectors, ~170 MB of memory instead of ~840. Your
# whole sandbox has 2.5 GB for every notebook you leave open, and a kernel you have
# forgotten about is still holding its share.
from langchain_core.embeddings import Embeddings

class MiniLMEmbeddings(Embeddings):
    """all-MiniLM-L6-v2 behind LangChain's Embeddings interface.

    Two methods is the whole contract -- which is why a store, a splitter and a chain
    never need to know which model is underneath, or what runtime it uses."""

    def __init__(self):
        from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2
        self._fn = ONNXMiniLM_L6_V2()

    def embed_documents(self, texts: list) -> list:
        return [[float(x) for x in v] for v in self._fn(list(texts))]

    def embed_query(self, text: str) -> list:
        return [float(x) for x in self._fn([text])[0]]


_emb_cache = {}
def get_embeddings():
    """The embedding model, built once per kernel."""
    if "model" not in _emb_cache:
        _emb_cache["model"] = MiniLMEmbeddings()
    return _emb_cache["model"]

print("work dir   :", WORK)
print("chat model :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

## Concept

A `Document` is two fields: `page_content` and `metadata`. Loaders produce them, splitters
cut them up, vector stores index them, retrievers hand them back. Everything downstream is
the same object.

The splitter has two numbers and both are decisions:

| | |
|---|---|
| `chunk_size` | too large and the vector averages several ideas, so it is close to nothing in particular. Too small and a rule gets separated from its exception |
| `chunk_overlap` | how much of the end of one chunk is repeated at the start of the next, so a sentence that straddles a cut is still findable |

`RecursiveCharacterTextSplitter` tries a list of separators in order &mdash; paragraphs
first, then lines, then spaces &mdash; so a paragraph survives whole wherever it can.

In [ ]:
# ------------------------------------------------- the raw document, written to disk
# The two sentences to watch are in CHAPTER 1: annual leave is 24 days, and unused leave
# cannot be carried forward. The second qualifies the first. Cut between them and no
# retriever on earth can return them together.

HANDBOOK_TEXT = """Employee Handbook

CHAPTER 1: LEAVE

Annual leave is 24 days per year for all full-time employees. Leave must be applied for at least 3 working days in advance through the HR portal. Unused annual leave cannot be carried forward to the next financial year.

Sick leave is 12 days per year. Notify your manager by 10 AM on the day of absence. A medical certificate is required for absences exceeding 2 consecutive days.

Maternity leave is 26 weeks of paid leave. Paternity leave is 2 weeks. Both must be applied for at least 30 days before the expected date.

CHAPTER 2: WORKING FROM HOME

Employees may work from home up to 3 days per week with team lead approval. Core hours are 10 AM to 4 PM IST and you must be reachable during them.

A VPN connection is mandatory for reaching internal systems from home. Contact the IT helpdesk for setup.

CHAPTER 3: EXPENSES

Travel expenses must be submitted with original receipts within 7 working days of travel. The meal allowance during client visits is 500 per day.

Internet reimbursement is 1,500 per month for employees working from home. Submit the broadband bill to finance by the 5th of each month.
"""

HANDBOOK_PATH = os.path.join(WORK, "handbook.txt")
with open(HANDBOOK_PATH, "w") as fh:
    fh.write(HANDBOOK_TEXT)

print(f"wrote {len(HANDBOOK_TEXT)} characters to {HANDBOOK_PATH}")

## Section 1 &mdash; A file becomes Documents

`TextLoader` returns a **list** of one `Document` for a text file. A PDF loader returns one
per page. Either way the next stage does not care, which is the point of the abstraction.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def load_handbook() -> list:
    """Given."""
    return TextLoader(HANDBOOK_PATH).load()


def extra_metadata() -> dict:
    """TextLoader records only `source`, and it records the temp path it read.

    A citation needs a name a human recognises, and a filter needs something to filter
    on. Add both: a `title` of "Employee Handbook" and a `doc_type` of "policy"."""
    return BLANK

In [ ]:
# --- Self-check: Section 1   (Document objects -- no store yet, no model)
check("TextLoader returns a list of Documents",
      lambda: isinstance(load_handbook(), list)
              and isinstance(load_handbook()[0], Document))
check("a text file loads as exactly one Document",
      lambda: len(load_handbook()) == 1,
      "a PDF loader would give you one per page -- same object either way")
check("the loader recorded where it came from",
      lambda: "source" in load_handbook()[0].metadata)
check("you added a human-readable title",
      lambda: extra_metadata()["title"] == "Employee Handbook")
check("and something a filter can use",
      lambda: extra_metadata()["doc_type"] == "policy")

## Section 2 &mdash; The cut

Now the decision. Section 1 of the handbook states the annual-leave allowance and then, two
sentences later, says unused leave cannot be carried forward. Someone asking
&ldquo;can I carry my leave over?&rdquo; needs both.

Pick a `chunk_size` that keeps that paragraph whole, and an overlap that keeps the boundary
recoverable.

In [ ]:
def chunk_size() -> int:
    """Big enough that the annual-leave paragraph survives in one piece, small enough that
    a chunk is still about one thing. The paragraph is a little over 200 characters."""
    return BLANK


def chunk_overlap() -> int:
    """How much of the end of one chunk to repeat at the start of the next. Roughly a
    sentence is the usual answer; zero is the usual mistake."""
    return BLANK


def split(docs: list) -> list:
    """Given -- your two numbers, wired into the splitter."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size(),
        chunk_overlap=chunk_overlap(),
        separators=["\n\n", "\n", " ", ""])
    return splitter.split_documents(docs)

In [ ]:
# --- Self-check: Section 2
def leave_chunk(chunks):
    """The chunk that states the 24-day allowance."""
    return next(c for c in chunks if "24 days" in c.page_content)

def chunks_now():
    return split(load_handbook())

check("the handbook splits into several chunks",
      lambda: 4 <= len(chunks_now()) <= 20)
check("chunk_size is a sane size for a policy paragraph",
      lambda: 200 <= chunk_size() <= 1000,
      "under 200 splits the paragraph; over 1000 and the vector means nothing in particular")
check("overlap is non-zero but not most of the chunk",
      lambda: 0 < chunk_overlap() <= chunk_size() // 4)
check("the allowance and the carry-forward rule are in ONE chunk",
      lambda: "carried forward" in leave_chunk(chunks_now()).page_content,
      "at a smaller chunk_size these separate, and the answer becomes unreachable")
check("every chunk kept the parent document's metadata",
      lambda: all("source" in c.metadata for c in chunks_now()))

def _cut_it_too_small():
    """The same document at chunk_size=120 -- the failure this lab is about."""
    small = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=0)
    chunks = small.split_documents(load_handbook())
    hit = next(c for c in chunks if "24 days" in c.page_content)
    print("  at 120 chars, the allowance chunk reads:")
    print("   ", repr(hit.page_content[:110]))
    print("   ...and 'carried forward' is in it:",
          "carried forward" in hit.page_content)
    print("    (it is still indexed -- just no longer attached to the rule it qualifies)")
guard(_cut_it_too_small)

In [ ]:
# --- What overlap actually buys, at the boundary
def _boundary():
    chunks = chunks_now()
    if len(chunks) < 2:
        print("(only one chunk -- lower chunk_size to see a boundary)")
        return
    tail = chunks[0].page_content[-chunk_overlap():]
    print("  end of chunk 1  :", repr(tail))
    print("  start of chunk 2:", repr(chunks[1].page_content[:chunk_overlap()]))
    print("  the tail reappears in chunk 2:", tail[:20] in chunks[1].page_content)

guard(_boundary)

def _sizes():
    for size in (120, 300, 800, 2000):
        s = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
        c = s.split_documents(load_handbook())
        avg = sum(len(x.page_content) for x in c) // len(c)
        print(f"  chunk_size={size:>5} -> {len(c):>2} chunks, {avg:>4} chars each")
guard(_sizes)

In [ ]:
score()

## Your turn

1. Set `chunk_overlap()` to 0 and re-run. Which self-check fails, and which one *should*
   have failed but did not? Overlap protects a boundary you have not thought of yet.
2. Drop `"\n\n"` from the separator list. The splitter now cuts on single newlines and
   never respects a paragraph. Look at the allowance chunk again.
3. Split by heading instead: `MarkdownHeaderTextSplitter` on `CHAPTER` lines keeps a whole
   chapter together and writes the heading into the metadata. On this document that is
   strictly better. On a 400-page contract it is strictly worse. Why?